# A Deep Dive into Bayesian Model Comparison: GARCH Models

This notebook demonstrates `ModelComparisonWorkflow` for discriminating between four GARCH-family models commonly used in computational finance.

| Model | Key feature |
|-------|-------------|
| ARCH(1) | Short-memory ARCH: volatility shocks decay in one step |
| GARCH(1,1) | Persistent volatility clustering, Gaussian innovations |
| GJR-GARCH(1,1) | Leverage effect: negative shocks inflate variance more |
| GARCH-t(1,1) | Persistent clustering with heavy-tailed Student-t innovations |

Rather than hand-crafting summary statistics, we feed the raw return series directly into a `TimeSeriesTransformer` summary network and let the network learn what to look at — a more principled and typically more powerful approach.

## Two families of scoring rules

BayesFlow supports two families of scoring rules for model comparison:

**PMP (Posterior Model Probability) rules** — the network outputs softmax probabilities over all models.  Diagnostics: confusion matrix and calibration curves.

| Class | Loss |
|-------|------|
| `CrossEntropyScore` | Categorical cross-entropy (default) |
| `SquaredScore` | Brier score (squared error on softmax probabilities) |
| `PolynomialScore` | Polynomial scoring rule; `alpha=2` recovers the Brier score |

**Bayes factor rules** — the network outputs M−1 log Bayes factors log K_{k,ref} relative to a reference model.  Diagnostics: blind coverage test.

| Class | Notes |
|-------|-------|
| `ExponentialScore` | Baseline Bayes factor rule |
| `LogisticScore` | Softplus-based; numerically more stable |
| `AlphaExponentialScore` | Exponential × polynomial envelope (default α=0.5) |
| `LPOPExponentialScore` | **Recommended** — l-POP transform before exp |
| `AlphaLogExponentialScore` | Log-sum-exp variant (default α=1.0) |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import bayesflow as bf
from bayesflow.simulators import Simulator
from bayesflow.workflows import ModelComparisonWorkflow
from bayesflow.scoring_rules import (
    CrossEntropyScore,
    SquaredScore,
    PolynomialScore,
    ExponentialScore,
    LogisticScore,
    AlphaExponentialScore,
    LPOPExponentialScore,
    AlphaLogExponentialScore,
)
import bayesflow.diagnostics.plots as bf_plots

## Simulators

Each simulator draws parameters from its prior, runs the GARCH recursion for `T + burn_in` steps, discards the burn-in, and returns the return series with shape `(batch_size, T, 1)`.  The trailing channel dimension is required by `TimeSeriesTransformer`. The `ModelComparisonSimulator` wrapping them will append `model_indices` automatically.

In [ ]:
class ARCH1Simulator(Simulator):
    """
    ARCH(1):  σ²_t = ω + α·ε²_{t-1},  ε_t = σ_t·z_t,  z_t ~ N(0, 1)

    No β term — volatility shocks dissipate after one step, producing short-memory
    volatility clustering that is qualitatively different from GARCH.

    Prior
    -----
    ω ~ U(5e-5, 2e-4),  α ~ U(0.10, 0.40)  [α < 1 ensures stationarity]
    μ ~ U(-5e-4, 5e-4)
    """

    def __init__(self, T: int = 250, burn_in: int = 50):
        self.T = T
        self.burn_in = burn_in

    def sample(self, batch_shape, **kwargs):
        B = int(np.prod(batch_shape))
        T_total = self.T + self.burn_in

        omega = np.random.uniform(5e-5, 2e-4, B)
        alpha = np.random.uniform(0.10, 0.40, B)
        mu    = np.random.uniform(-5e-4, 5e-4, B)

        returns = np.empty((B, T_total))
        eps     = np.zeros(B)
        sigma2  = omega / np.maximum(1.0 - alpha, 1e-6)

        for t in range(T_total):
            sigma2        = omega + alpha * eps ** 2
            sigma2        = np.maximum(sigma2, 1e-10)
            eps           = np.sqrt(sigma2) * np.random.randn(B)
            returns[:, t] = mu + eps

        return {"returns": returns[:, self.burn_in:, np.newaxis].astype(np.float32)}

In [ ]:
class GARCH11Simulator(Simulator):
    """
    GARCH(1,1):  σ²_t = ω + α·ε²_{t-1} + β·σ²_{t-1},  ε_t = σ_t·z_t,  z_t ~ N(0, 1)

    Prior
    -----
    ω ~ U(1e-6, 5e-5),  α ~ U(0.05, 0.15),  β ~ U(0.75, 0.84)  [α+β ≤ 0.99]
    μ ~ U(-5e-4, 5e-4)
    """

    def __init__(self, T: int = 250, burn_in: int = 50):
        self.T = T
        self.burn_in = burn_in

    def sample(self, batch_shape, **kwargs):
        B = int(np.prod(batch_shape))
        T_total = self.T + self.burn_in

        omega = np.random.uniform(1e-6, 5e-5, B)
        alpha = np.random.uniform(0.05, 0.15, B)
        beta  = np.random.uniform(0.75, 0.84, B)
        mu    = np.random.uniform(-5e-4, 5e-4, B)

        returns = np.empty((B, T_total))
        eps     = np.zeros(B)
        sigma2  = omega / np.maximum(1.0 - alpha - beta, 1e-6)

        for t in range(T_total):
            sigma2        = omega + alpha * eps ** 2 + beta * sigma2
            sigma2        = np.maximum(sigma2, 1e-10)
            eps           = np.sqrt(sigma2) * np.random.randn(B)
            returns[:, t] = mu + eps

        return {"returns": returns[:, self.burn_in:, np.newaxis].astype(np.float32)}

In [ ]:
class GJRGARCHSimulator(Simulator):
    """
    GJR-GARCH(1,1):  σ²_t = ω + (α + γ·I_{t-1})·ε²_{t-1} + β·σ²_{t-1}
    I_{t-1} = 1 if ε_{t-1} < 0  (Glosten, Jagannathan & Runkle 1993)

    γ > 0 means negative shocks increase volatility more than positive ones.
    Stationarity requires α + γ/2 + β < 1.

    Prior
    -----
    ω ~ U(1e-6, 5e-5),  α ~ U(0.02, 0.08),  γ ~ U(0.05, 0.12),
    β ~ U(0.75, 0.85)   [α + γ/2 + β ≤ 0.99],  μ ~ U(-5e-4, 5e-4)
    """

    def __init__(self, T: int = 250, burn_in: int = 50):
        self.T = T
        self.burn_in = burn_in

    def sample(self, batch_shape, **kwargs):
        B = int(np.prod(batch_shape))
        T_total = self.T + self.burn_in

        omega = np.random.uniform(1e-6, 5e-5, B)
        alpha = np.random.uniform(0.02, 0.08, B)
        gamma = np.random.uniform(0.05, 0.12, B)
        beta  = np.random.uniform(0.75, 0.85, B)
        mu    = np.random.uniform(-5e-4, 5e-4, B)

        returns = np.empty((B, T_total))
        eps     = np.zeros(B)
        sigma2  = omega / np.maximum(1.0 - alpha - gamma / 2.0 - beta, 1e-6)

        for t in range(T_total):
            I_neg         = (eps < 0).astype(float)
            sigma2        = omega + (alpha + gamma * I_neg) * eps ** 2 + beta * sigma2
            sigma2        = np.maximum(sigma2, 1e-10)
            eps           = np.sqrt(sigma2) * np.random.randn(B)
            returns[:, t] = mu + eps

        return {"returns": returns[:, self.burn_in:, np.newaxis].astype(np.float32)}

In [ ]:
class GARCHtSimulator(Simulator):
    """
    GARCH-t(1,1):  same volatility dynamics as GARCH(1,1), but
    z_t ~ t_ν / √(ν/(ν−2))  so that Var(z_t) = 1.

    Prior
    -----
    ω ~ U(1e-6, 5e-5),  α ~ U(0.05, 0.15),  β ~ U(0.75, 0.84)
    ν ~ U(4, 15)  (degrees of freedom; lower ↔ heavier tails)
    """

    def __init__(self, T: int = 250, burn_in: int = 50):
        self.T = T
        self.burn_in = burn_in

    def sample(self, batch_shape, **kwargs):
        B = int(np.prod(batch_shape))
        T_total = self.T + self.burn_in

        omega = np.random.uniform(1e-6, 5e-5, B)
        alpha = np.random.uniform(0.05, 0.15, B)
        beta  = np.random.uniform(0.75, 0.84, B)
        mu    = np.random.uniform(-5e-4, 5e-4, B)
        nu    = np.random.uniform(4.0, 15.0, B)
        scale = np.sqrt((nu - 2.0) / nu)

        returns = np.empty((B, T_total))
        eps     = np.zeros(B)
        sigma2  = omega / np.maximum(1.0 - alpha - beta, 1e-6)

        for t in range(T_total):
            sigma2        = omega + alpha * eps ** 2 + beta * sigma2
            sigma2        = np.maximum(sigma2, 1e-10)
            z             = np.random.standard_t(nu) * scale
            eps           = np.sqrt(sigma2) * z
            returns[:, t] = mu + eps

        return {"returns": returns[:, self.burn_in:, np.newaxis].astype(np.float32)}

In [ ]:
T = 250
MODEL_NAMES = ["ARCH(1)", "GARCH(1,1)", "GJR-GARCH(1,1)", "GARCH-t(1,1)"]

simulators = [
    ARCH1Simulator(T=T),
    GARCH11Simulator(T=T),
    GJRGARCHSimulator(T=T),
    GARCHtSimulator(T=T),
]

### Prior predictive check

A single draw from each simulator illustrates the qualitative differences the network will learn to detect.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=False)
for ax, sim, name in zip(axes, simulators, MODEL_NAMES):
    path = sim.sample((1,))["returns"][0, :, 0]
    ax.plot(path, linewidth=0.6, color="#132a70")
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Day")
    ax.axhline(0, color="gray", linewidth=0.4, linestyle="--")
plt.suptitle("Prior predictive: example return paths", y=1.02)
plt.tight_layout()
plt.show()

---

## Part 1 — PMP Scoring Rules

PMP scoring rules train the network to output softmax probabilities over all models. The three available rules differ only in their loss function; they share the same network architecture and produce identical output shapes.

We train with the default `CrossEntropyScore`.  To switch to another PMP rule, pass `scoring_rule=SquaredScore()` or `scoring_rule=PolynomialScore(alpha=3.0)` — everything else stays the same.

In [ ]:
pmp_workflow = ModelComparisonWorkflow(
    simulator=simulators,
    summary_variables=["returns"],
    summary_network="time_series_transformer",
    model_names=MODEL_NAMES,
    scoring_rule=CrossEntropyScore(),  # default; may be omitted
)

# Alternative PMP scoring rules (same API, just swap scoring_rule=):
#
# pmp_workflow = ModelComparisonWorkflow(
#     ..., scoring_rule=SquaredScore()
# )
#
# pmp_workflow = ModelComparisonWorkflow(
#     ..., scoring_rule=PolynomialScore(alpha=2.0)  # alpha=2 recovers Brier score
# )

In [ ]:
history_pmp = pmp_workflow.fit_online(
    epochs=30,
    num_batches_per_epoch=100,
    batch_size=32,
)

### Diagnostics — PMP rules

`plot_default_diagnostics` detects the active scoring rule and automatically produces the appropriate plots.  For PMP rules these are:

- **Loss curve** — training and validation cross-entropy over epochs.
- **Confusion matrix** — rows = true model, columns = predicted model (argmax of PMPs).
- **Calibration curves** — per-model ECE-annotated reliability diagrams.

In [ ]:
figs_pmp = pmp_workflow.plot_default_diagnostics(test_data=500)
for name, fig in figs_pmp.items():
    plt.show()

## Part 2 — Bayes Factor Scoring Rules

Bayes factor scoring rules train the network to output **log Bayes factors** log K_{k,0} for each competing model k relative to the reference model (model 0 by default). The output shape is `(num_datasets, num_models - 1)`.

All five rules share the same network structure and diagnostics; they differ in how they penalize incorrect rankings:

| Rule | Key property |
|------|--------------|
| `ExponentialScore` | Baseline; may be unstable when log BF magnitudes are large |
| `LogisticScore` | Softplus penalty; numerically more stable than exponential |
| `AlphaExponentialScore(alpha)` | Exponential × polynomial envelope; α=0.5 default |
| `LPOPExponentialScore(alpha)` | **Recommended** — l-POP transform regularises large outputs |
| `AlphaLogExponentialScore(alpha)` | Log-sum-exp variant; interpolates between max and sum |

We train with `LPOPExponentialScore`, which applies the leaky parity-odd power transform $J_\alpha(x) = x + x|x|^{(\alpha−1)}$ before the exponential, giving well-calibrated estimates even when true log Bayes factors are large.

In [ ]:
bayes_factor_workflow = ModelComparisonWorkflow(
    simulator=simulators,
    summary_variables=["returns"],
    summary_network="time_series_transformer",
    model_names=MODEL_NAMES,
    scoring_rule=LPOPExponentialScore(alpha=2.0),
)

# Other Bayes factor scoring rules (same API, just swap scoring_rule=):
#
# bayes_factor_workflow = ModelComparisonWorkflow(
#     ..., scoring_rule=ExponentialScore()
# )
#
# bayes_factor_workflow = ModelComparisonWorkflow(
#     ..., scoring_rule=LogisticScore()
# )
#
# bayes_factor_workflow = ModelComparisonWorkflow(
#     ..., scoring_rule=AlphaExponentialScore(alpha=0.5)
# )
#
# bayes_factor_workflow = ModelComparisonWorkflow(
#     ..., scoring_rule=AlphaLogExponentialScore(alpha=1.0)
# )

In [ ]:
history_bayes_factor = bayes_factor_workflow.fit_online(
    epochs=30,
    num_batches_per_epoch=100,
    batch_size=32,
)

### Diagnostics — Bayes factor rules

`plot_default_diagnostics` detects the Bayes factor rule and automatically switches to
the **blind coverage test** (Jeffrey & Wandelt 2024) instead of the PMP diagnostics.

For PMP rules these are:

- **Loss curve** — training and validation loss over epochs.
- **Blind coverage** — one panel per competing model.  For each marginal quantile
  level α the plot shows what fraction of each model group's predicted log K falls at
  or below the α-quantile of the *full* (model-label-free) predicted log K
  distribution.  A well-calibrated estimator pushes the reference-model curve *above*
  the diagonal and the competing-model curve *below* it.

In [ ]:
figs_bayes_factor = bayes_factor_workflow.plot_default_diagnostics(test_data=500)
for name, fig in figs_bayes_factor.items():
    plt.show()

### Bayes factor recovery via importance sampling

`bayes_factor_recovery` requires ground-truth log Bayes factors.  GARCH models have no analytic marginal likelihood, but the likelihood *is* evaluable in closed form given parameters and data — which is all importance sampling needs.

Drawing S parameter vectors from the prior and averaging the resulting likelihoods gives a consistent estimator of the log marginal likelihood for each model:

$$\log \hat{p}(\mathbf{y} \mid M_k) = \log\!\sum_{s=1}^{S} p\!\left(\mathbf{y} \mid \theta_s^{(k)},\, M_k\right) - \log S, \qquad \theta_s^{(k)} \sim p(\theta \mid M_k)$$

The estimated log Bayes factor is then $\log \hat{K}_{k,0} = \log \hat{p}(\mathbf{y} \mid M_k) - \log \hat{p}(\mathbf{y} \mid M_0)$.

The estimator is implemented below with vectorisation over the S-dimension: at each time step, all S log-likelihoods and all B test datasets are updated simultaneously, so the only sequential loop runs over T=250 time steps.

In [ ]:
from scipy.special import logsumexp, gammaln


def _gauss_logpdf(x, sigma2):
    return -0.5 * (np.log(2 * np.pi) + np.log(sigma2) + x**2 / sigma2)


def _t_logpdf(x, nu, sigma2):
    scale2 = sigma2 * (nu - 2.0) / nu
    return (gammaln((nu + 1) / 2) - gammaln(nu / 2)
            - 0.5 * np.log(np.pi * nu * scale2)
            - (nu + 1) / 2 * np.log(1.0 + x**2 / (nu * scale2)))


def _arch1_log_ml(y: np.ndarray, S: int) -> np.ndarray:
    B, T = y.shape
    omega = np.random.uniform(5e-5, 2e-4, S)[:, None]
    alpha = np.random.uniform(0.10, 0.40, S)[:, None]
    mu    = np.random.uniform(-5e-4, 5e-4, S)[:, None]

    log_liks = np.zeros((S, B))
    sigma2 = (omega / np.maximum(1.0 - alpha, 1e-6)) * np.ones((S, B))
    for t in range(T):
        et = y[None, :, t] - mu
        sigma2 = np.maximum(sigma2, 1e-10)
        log_liks += _gauss_logpdf(et, sigma2)
        sigma2 = omega + alpha * et**2
    return logsumexp(log_liks, axis=0) - np.log(S)


def _garch11_log_ml(y: np.ndarray, S: int) -> np.ndarray:
    B, T = y.shape
    omega = np.random.uniform(1e-6, 5e-5, S)[:, None]
    alpha = np.random.uniform(0.05, 0.15, S)[:, None]
    beta  = np.random.uniform(0.75, 0.84, S)[:, None]
    mu    = np.random.uniform(-5e-4, 5e-4, S)[:, None]

    log_liks = np.zeros((S, B))
    sigma2 = (omega / np.maximum(1.0 - alpha - beta, 1e-6)) * np.ones((S, B))
    for t in range(T):
        et = y[None, :, t] - mu
        sigma2 = np.maximum(sigma2, 1e-10)
        log_liks += _gauss_logpdf(et, sigma2)
        sigma2 = omega + alpha * et**2 + beta * sigma2
    return logsumexp(log_liks, axis=0) - np.log(S)


def _gjrgarch_log_ml(y: np.ndarray, S: int) -> np.ndarray:
    B, T = y.shape
    omega = np.random.uniform(1e-6, 5e-5, S)[:, None]
    alpha = np.random.uniform(0.02, 0.08, S)[:, None]
    gamma = np.random.uniform(0.05, 0.12, S)[:, None]
    beta  = np.random.uniform(0.75, 0.85, S)[:, None]
    mu    = np.random.uniform(-5e-4, 5e-4, S)[:, None]

    log_liks = np.zeros((S, B))
    sigma2 = (omega / np.maximum(1.0 - alpha - gamma / 2.0 - beta, 1e-6)) * np.ones((S, B))
    for t in range(T):
        et = y[None, :, t] - mu
        sigma2 = np.maximum(sigma2, 1e-10)
        log_liks += _gauss_logpdf(et, sigma2)
        sigma2 = omega + (alpha + gamma * (et < 0).astype(float)) * et**2 + beta * sigma2
    return logsumexp(log_liks, axis=0) - np.log(S)


def _garcht_log_ml(y: np.ndarray, S: int) -> np.ndarray:
    B, T = y.shape
    omega = np.random.uniform(1e-6, 5e-5, S)[:, None]
    alpha = np.random.uniform(0.05, 0.15, S)[:, None]
    beta  = np.random.uniform(0.75, 0.84, S)[:, None]
    mu    = np.random.uniform(-5e-4, 5e-4, S)[:, None]
    nu    = np.random.uniform(4.0, 15.0, S)[:, None]

    log_liks = np.zeros((S, B))
    sigma2 = (omega / np.maximum(1.0 - alpha - beta, 1e-6)) * np.ones((S, B))
    for t in range(T):
        et = y[None, :, t] - mu
        sigma2 = np.maximum(sigma2, 1e-10)
        log_liks += _t_logpdf(et, nu, sigma2)
        sigma2 = omega + alpha * et**2 + beta * sigma2
    return logsumexp(log_liks, axis=0) - np.log(S)


def compute_garch_log_bfs(returns: np.ndarray, S: int = 1000, reference: int = 0) -> np.ndarray:
    """Approximate log K_{k, reference} for all four GARCH models via prior IS.

    Parameters
    ----------
    returns : (B, T) array of return series.
    S : IS sample count per model.  Higher reduces variance; 1000 is a reasonable default.
    reference : index of the reference model (0 = ARCH(1)).

    Returns
    -------
    log_bfs : (B, 3) array of log Bayes factors for the three competing models.
    """
    fns = [_arch1_log_ml, _garch11_log_ml, _gjrgarch_log_ml, _garcht_log_ml]
    log_mls = np.stack([fn(returns, S) for fn in fns], axis=1)   # (B, 4)
    competing = [k for k in range(4) if k != reference]
    return log_mls[:, competing] - log_mls[:, reference : reference + 1]  # (B, 3)

In [ ]:
test_data_garch    = bayes_factor_workflow.simulate(500)
pred_log_bfs_garch = bayes_factor_workflow.predict(conditions=test_data_garch)  # (500, 3)

returns_test       = test_data_garch["returns"][:, :, 0]           # (500, 250)
true_log_bfs_garch = compute_garch_log_bfs(returns_test, S=1000)   # (500, 3); takes ~30 s

fig = bf_plots.bayes_factor_recovery(
    pred_log_bayes_factors=pred_log_bfs_garch,
    true_log_bayes_factors=true_log_bfs_garch,
    true_models=test_data_garch["model_indices"],
    model_names=MODEL_NAMES,
    reference_model=0,
    add_corr=True,
)
plt.show()